# Linear Regression

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/Users/shailesh/Desktop/vedam_Sem3_ml/Data/mpg.csv')
df.info()

---
## 1. Where we are

> **Flow:** KNN predicted a label. Now we predict a number.

Last class the target was `origin` — one of three labels.
Today the target is `mpg` — a number in a range.

---
## 2. What comes next

> **Flow:** Two families of supervised algorithms.

| Regression | Classification |
|---|---|
| **Linear Regression** ← today | KNN ✓ done |
| Ridge / Lasso | Logistic Regression |
| Decision Tree Regressor | Decision Tree |
| Random Forest Regressor | Random Forest |
| KNN Regressor  | |

KNN appears on both sides. If the target is a label the neighbours vote.
If it is a number the neighbours are averaged.

### Practice 2

**Q1.** Amazon wants to predict how many days a delivery will take.
Regression or classification?

**Q2.** Ola wants to predict which of four ride types a customer will book.
Regression or classification?

<details><summary>Answers</summary>

1. Regression — days is a number in a range.
2. Classification — four labels.
</details>

---
## 3. The idea

> **Flow:** A line that passes as close as possible to the points.

**It is the line from school.** `y = mx + c`, with new names:

```
mpg = coef × weight + intercept
```

**Many lines can be drawn.** Only one fits best.

**Residual = actual − predicted.** The vertical distance from a point to the line.

**The best line has the smallest total of squared residuals.**
Squaring stops +4 and −4 cancelling into 0.

**With many features** the line cannot be drawn any more, but the job is the same:

```
mpg = w1×weight + w2×horsepower + ... + intercept
```

### Practice 3

**Q1.** A model predicts 22 for a car whose actual mileage is 26. What is the residual?

**Q2.** One residual is +4 and another is −4. If we just added them, what would the total be,
and why is that a problem?

<details><summary>Answers</summary>

1. 26 − 22 = 4.
2. Zero. A model that is wrong twice looks perfect. Squaring makes both count as 16.
</details>

---
## 4. Load, clean, split

> **Flow:** Get the data ready. Nothing clever today.

In [ ]:
df.head(2)

In [ ]:
df = df.drop(columns=['name', 'origin'])
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
print(df.shape, "|", df.isna().sum().sum(), "missing")
df.head()

Two columns dropped:

- `name` — unique for every car, so there is no pattern to learn.
- `origin` — text. Turning it into numbers needs an encoder, and today is about the model,
  not the preprocessing. It comes back next class.

Everything left is a number.

In [ ]:
X = df.drop(columns='mpg')
y = df['mpg']
print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 
)

print(X_train.shape, X_test.shape)

No `stratify` this time — that splits a label evenly, and our target is a number.

---
## 5. Look at the data first

> **Flow:** Before fitting anything, see whether there is a pattern at all.

In [ ]:

sns.scatterplot(data=df, x='weight', y='mpg')
plt.title('weight vs mpg')
plt.show()

The points fall from top-left to bottom-right. Heavier cars get lower mileage.

The pattern is there. Now we need one line that describes it.

---
## 6. One feature

> **Flow:** `weight` in, `mpg` out.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train[['weight']], y_train)
print("coef:   ", model.coef_)
print("intercept:", model.intercept_)

```
mpg = -0.0078 × weight + 46.78
```

The coefficient is negative — heavier car, lower mileage.
That one number is the entire model.

In [ ]:
print(model.score(X_test[['weight']], y_test)) # r2 score 

In [ ]:
sns.scatterplot(x=X_test['weight'], y=y_test, label='actual')
sns.lineplot(x=X_test['weight'], y=model.predict(X_test[['weight']]),
             color='red', label='fitted line')
plt.show()

---
## 7. All features

> **Flow:** Same model, more columns. No scaling, no encoding.

In [ ]:
model_all = LinearRegression()
model_all.fit(X_train, y_train)
print(model_all.score(X_test, y_test))

One feature gave 0.723. All six give 0.824.


**A note on scaling.** Last class KNN needed it badly. Linear regression does not — it
measures no distances. Scaling only changes the units the coefficients are written in,
not the predictions.

---
## 8. Metrics

> **Flow:** Accuracy does not work here. The answer is a number, not a label.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model_all.predict(X_test)

print("MAE: ", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:  ", r2_score(y_test, y_pred))

| Metric | Value | Meaning |
|---|---|---|
| MAE | 2.47 | on average, off by 2.47 mpg |
| RMSE | 3.07 | large errors count more |
| R² | 0.824 | 0 = no better than guessing the average, 1 = perfect |

RMSE is always greater than or equal to MAE.

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.2, random_state=7)
print(LinearRegression().fit(X_tr2, y_tr2).score(X_te2, y_te2))

Same data, same model, different split — different score.
We come back to this in a later class.

### Practice 8

**Q1.** A house price model reports MAE = 2.5 lakh. What does that mean in one line?

**Q2.** Model A has R² = 0.91, model B has R² = 0.40. Which is better, and what does
R² = 0 mean?

<details><summary>Answers</summary>

1. On average the predicted price is off by 2.5 lakh.
2. A. R² = 0 means no better than always guessing the average price.
</details>

---
# Quiz — check yourself



**Q1.** A food delivery app wants to predict how many minutes an order will take to arrive.
Regression or classification?

<details><summary>Answer</summary>

Regression — minutes is a number in a range, not one label from a fixed set.
</details>

**Q2.** A model predicts 28 for a car whose actual mileage is 25. What is the residual?

<details><summary>Answer</summary>

residual = actual − predicted = 25 − 28 = **−3**

The sign matters. Negative means the model predicted too high.
</details>

**Q3.** Why do we square the residuals instead of just adding them up?

<details><summary>Answer</summary>

+4 and −4 would cancel out to 0, and a model that was wrong twice would look perfect.
Squaring makes both count as 16.
</details>

**Q4.** The coefficient for `weight` came out negative. What does that mean?

<details><summary>Answer</summary>

Heavier cars get lower mileage. As the feature goes up, the prediction goes down.
</details>

**Q5.** A house price model reports MAE = 3 lakh. What does that number mean?

<details><summary>Answer</summary>

On average, the predicted price is off by about 3 lakh.
MAE is in the same units as the target, which is what makes it readable.
</details>

**Q6.** Can RMSE ever be smaller than MAE?

<details><summary>Answer</summary>

No, never. Squaring inflates the large errors before the average is taken, so RMSE is
always greater than or equal to MAE.
</details>

**Q7.** A model reports R&#178; = 0. What does that tell you?

<details><summary>Answer</summary>

The model is no better than always guessing the average of the target.
It is not broken — it is just useless.
</details>

---
# Does scaling help linear regression?

> **Flow:** Last class scaling changed KNN a lot. Let us test it here.

Same data, same model. The only change is that every column is put on the same scale.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print(X_train_scaled.shape, X_test_scaled.shape)

(318, 6) (80, 6)


In [23]:
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)
print("without scaling:", model_all.score(X_test, y_test))
print("with scaling:   ", model_scaled.score(X_test_scaled, y_test))

without scaling: 0.824424533094336
with scaling:    0.8244245330943358


Both are **0.8244**. Not close — identical.

In [24]:
pd.DataFrame({
    'feature': X_train.columns,
    'coef without scaling': model_all.coef_.round(3),
    'coef with scaling': model_scaled.coef_.round(3)
})

,feature,coef without scaling,coef with scaling
0,cylinders,0.069,0.116
1,displacement,0.002,0.165
2,horsepower,0.003,0.107
3,weight,-0.007,-5.912
4,acceleration,0.081,0.222
5,model_year,0.801,2.883


| feature | without scaling | with scaling |
|---|---|---|
| cylinders | 0.069 | 0.116 |
| displacement | 0.002 | 0.165 |
| horsepower | 0.003 | 0.107 |
| weight | −0.007 | **−5.912** |
| acceleration | 0.081 | 0.222 |
| model_year | 0.801 | **2.883** |

**The coefficients changed. The predictions did not.**

Unscaled, `weight` moves in steps of about 1 kg, so its coefficient is tiny — −0.007 per
kilogram. Scaled, `weight` moves in steps of one standard deviation, so the same
relationship is now written as −5.912. The model is saying exactly the same thing in
different units.

**Why scaling does nothing here:** linear regression never measures the distance between
two rows. It fits one line and reads off a value. Changing the units of a column changes
the coefficient that multiplies it, and the two changes cancel out exactly.

KNN is the opposite. Its whole prediction *is* a distance calculation, so the column with
the biggest numbers decides everything.

One useful side effect: scaled coefficients **are** comparable to each other. Unscaled,
you cannot say `model_year` (0.801) matters more than `weight` (−0.007) — the units are
different. Scaled, you can, and `weight` is clearly the strongest.

---
# KNN Regressor — the same problem, a different algorithm

> **Flow:** KNN can predict numbers too. Instead of voting, it averages its neighbours.

Same `X_train`, `X_test`, `y_train`, `y_test`. Only the model changes.

### Without scaling

In [25]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor()
knn.fit(X_train, y_train)
acc_knn_raw = knn.score(X_test, y_test)
print(acc_knn_raw)

0.7659996807953288


### With scaling

In [26]:
knn_scaled = KNeighborsRegressor()
knn_scaled.fit(X_train_scaled, y_train)
acc_knn_scaled = knn_scaled.score(X_test_scaled, y_test)
print(acc_knn_scaled)

0.8785489694954951


### All three together

In [ ]:
pd.DataFrame({
    'model': ['Linear Regression',
              'KNN Regressor (no scaling)',
              'KNN Regressor (scaled)'],
    'R2': [round(model_all.score(X_test, y_test), 4),
           round(acc_knn_raw, 4),
           round(acc_knn_scaled, 4)]
})


,model,R2
0,Linear Regression,0.8244
1,KNN Regressor (no scaling),0.7660
2,KNN Regressor (scaled),0.8785


| model | R&#178; |
|---|---|
| Linear Regression | 0.8244 |
| KNN Regressor, no scaling | 0.7660 |
| KNN Regressor, scaled | **0.8785** |

Two things worth taking away.

**Scaling gave KNN about 11 points and linear regression exactly 0.** The same step is
essential for one model and pointless for the other. It depends on how the model works,
not on how good the model is.

**The scaled KNN beat today's new model.** A newer or more complex algorithm is not
automatically better. A simple model that has been prepared properly often wins.

In [ ]:
from sklearn.metrics import mean_absolute_error

print("Linear Regression MAE:", mean_absolute_error(y_test, model_all.predict(X_test)))
print("KNN (scaled) MAE:     ", mean_absolute_error(y_test, knn_scaled.predict(X_test_scaled)))

### Practice

**Q1.** We fitted the scaler with `fit_transform` on the training data and only
`transform` on the test data. What would go wrong if we used `fit_transform` on both?

<details><summary>Answer</summary>

The scaler would learn the mean and spread of the test rows too. Information about the
test set would leak into the preparation, and the final score would be flattering rather
than honest.
</details>

**Q2.** KNN scored higher here. Does that mean we should always use KNN instead of linear
regression?

<details><summary>Answer</summary>

No. It won on this dataset, with this split, at k=5. Linear regression gives us
coefficients we can read and explain, runs instantly on large data, and does not need
scaling. KNN gives none of that. Which one to use depends on what you need, not on one
score.
</details>